# 三数据集 × 五模型 × 五步长训练

这个 notebook 用于直接训练完整实验矩阵：

- 数据集：ETTh1、ETTm1、ECL
- 预测步长：24、48、96、168、336
- 模型：LSTM、Transformer、Informer、Autoformer、PatchTST

训练按单个实验保存结果。中途手动停止后，重新运行本 notebook 会根据 `summary.json` 和 `results.npy` 跳过已完成实验，只继续未完成项。

## 0. 配置

`sample_limit=0` 表示使用完整样本；如果只想先做烟雾测试，可以改成 `64` 或 `512`。当前仓库中 ECL 预处理文件可能不存在，默认不会自动生成，避免误触发大规模预处理；确认要补齐 ECL 时，把 `PREPARE_MISSING_DATA` 改为 `True`。

In [ ]:
from pathlib import Path
from types import SimpleNamespace
from collections import defaultdict
import json
import sys
import time

# 定位项目根目录：支持从项目根目录或 notebooks/ 中启动 Jupyter
ROOT = Path.cwd().resolve()
for _ in range(6):
    if (ROOT / 'scripts').is_dir() and (ROOT / 'models').is_dir():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print(f'Project root: {ROOT}')

RUN_CONFIG = {
    'datasets': 'ETTh1,ETTm1,ECL',
    'horizons': '24,48,96,168,336',
    'models': 'lstm,transformer,informer,autoformer,patchtst',
    # 全局默认值只作为兜底；正式训练会在第 3 节按 MODEL_CONFIGS 为每个模型覆盖。
    'epochs': 50,
    'patience': 10,
    'batch_size': 32,
    'lr': 1e-3,
    'weight_decay': 1e-5,
    'device': 'auto',
    'seed': 42,
    'run_tag': 'full_val_best_e50p10_tb_seed42',
    'data_dir': 'data/processed',
    'sample_limit': 0,
    'num_workers': 0,
    'no_tensorboard': False,
    'skip_existing': True,
}

# 五个模型均使用 ETTh1 h96 超参数搜索中 best_val_loss 最低的配置。
# 来源：test_results/h96/ETTh1/{model}/*_summary.json。
MODEL_CONFIGS = {
    'lstm': {
        'source_experiment': 'test_results/h96/ETTh1/lstm/ETTh1_h96_lstm_h256_l1_dp01_lr0.001_wd0.0_summary.json',
        'selection_metric': {'best_val_loss': 0.8898224516826517, 'best_val_r2': 0.3434822692590601},
        'model': {
            'hidden_size': 256,
            'num_layers': 1,
            'dropout': 0.1,
        },
        'training': {
            'epochs': 50,
            'patience': 10,
            'batch_size': 128,
            'lr': 0.001,
            'weight_decay': 0.0,
        },
    },
    'transformer': {
        'source_experiment': 'test_results/h96/ETTh1/transformer/ETTh1_h96_transformer_d128_h4_l2_ff128_dp01_lr0.0001_wd0.0_summary.json',
        'selection_metric': {'best_val_loss': 0.9057894443764406, 'best_val_r2': 0.30944681097479426},
        'model': {
            'd_model': 128,
            'nhead': 4,
            'num_layers': 2,
            'dim_feedforward': 128,
            'dropout': 0.1,
        },
        'training': {
            'epochs': 50,
            'patience': 10,
            'batch_size': 128,
            'lr': 0.0001,
            'weight_decay': 0.0,
        },
    },
    'informer': {
        'source_experiment': 'test_results/h96/ETTh1/informer/ETTh1_h96_informer_d64_h4_enc2_dec2_ff256_fac3_dp01_summary.json',
        'selection_metric': {'best_val_loss': 0.8180458586005603, 'best_val_r2': 0.3553355616681716},
        'model': {
            'd_model': 64,
            'n_heads': 4,
            'n_encoder_layers': 2,
            'n_decoder_layers': 2,
            'd_ff': 256,
            'factor': 3,
            'dropout': 0.1,
        },
        'training': {
            'epochs': 50,
            'patience': 10,
            'batch_size': 128,
            'lr': 0.001,
            'weight_decay': 1e-5,
        },
    },
    'autoformer': {
        'source_experiment': 'test_results/h96/ETTh1/autoformer/ETTh1_h96_autoformer_d64_h4_enc2_dec1_ff128_fac3_ks25_summary.json',
        'selection_metric': {'best_val_loss': 0.6663595385411206, 'best_val_r2': 0.45639897795284495},
        'model': {
            'd_model': 64,
            'n_heads': 4,
            'n_encoder_layers': 2,
            'n_decoder_layers': 1,
            'd_ff': 128,
            'factor': 3,
            'dropout': 0.1,
            'kernel_size': 25,
        },
        'training': {
            'epochs': 50,
            'patience': 10,
            'batch_size': 128,
            'lr': 0.001,
            'weight_decay': 1e-5,
        },
    },
    'patchtst': {
        'source_experiment': 'test_results/h96/ETTh1/patchtst/ETTh1_h96_patchtst_d64_h8_l2_ff128_pl32_st8_dp01_summary.json',
        'selection_metric': {'best_val_loss': 0.6808768481016159, 'best_val_r2': 0.46077433333677403},
        'model': {
            'd_model': 64,
            'n_heads': 8,
            'n_layers': 2,
            'd_ff': 128,
            'patch_len': 32,
            'stride': 8,
            'dropout': 0.1,
        },
        'training': {
            'epochs': 50,
            'patience': 10,
            'batch_size': 128,
            'lr': 0.001,
            'weight_decay': 1e-5,
        },
    },
}

# 如果缺失预处理数据，是否在 notebook 中自动补齐。
# ECL 全量预处理与训练都比较重，默认 False 更安全。
PREPARE_MISSING_DATA = False
PREPROCESS_MAX_SAMPLES_PER_SPLIT = 0 # 0 表示预处理完整样本；烟雾测试可改为 512/2048
MISSING_DATA_POLICY = 'skip'  # 'skip' 或 'raise'
RUN_ONLY_FIRST_N = None     # 调试时可设为 1；正式运行保持 None

RUN_CONFIG, MODEL_CONFIGS

## 1. 检查训练矩阵与数据文件

这一节只检查，不训练。`completed` 表示已经有完整结果；`missing_data` 表示缺少对应数据集/步长的 `train.npz`、`val.npz` 或 `test.npz`。

In [ ]:
from scripts.run_experiments import MODEL_BUILDERS, parse_csv_list, parse_int_list
from models import AutoformerModel, InformerModel, LSTMModel, PatchTSTModel, TransformerModel

def register_validation_best_builders():
    model_classes = {
        'lstm': LSTMModel,
        'transformer': TransformerModel,
        'informer': InformerModel,
        'autoformer': AutoformerModel,
        'patchtst': PatchTSTModel,
    }
    for model_name, cfg in MODEL_CONFIGS.items():
        model_cls = model_classes[model_name]
        model_kwargs = dict(cfg['model'])
        MODEL_BUILDERS[model_name] = (
            lambda input_size, horizon, model_cls=model_cls, model_kwargs=model_kwargs:
            model_cls(input_size=input_size, horizon=horizon, **model_kwargs)
        )

register_validation_best_builders()

datasets = parse_csv_list(RUN_CONFIG['datasets'])
horizons = parse_int_list(RUN_CONFIG['horizons'])
models = parse_csv_list(RUN_CONFIG['models'])
unknown_models = sorted(set(models) - set(MODEL_BUILDERS))
if unknown_models:
    raise ValueError(f'Unknown models: {unknown_models}')
missing_model_configs = sorted(set(models) - set(MODEL_CONFIGS))
if missing_model_configs:
    raise ValueError(f'Missing MODEL_CONFIGS entries: {missing_model_configs}')

data_dir = Path(RUN_CONFIG['data_dir'])
if not data_dir.is_absolute():
    data_dir = ROOT / data_dir

run_tag_dir = RUN_CONFIG['run_tag'] or 'default'

def required_data_files(dataset_name, horizon):
    h_dir = data_dir / dataset_name / f'h{horizon}'
    return [h_dir / 'train.npz', h_dir / 'val.npz', h_dir / 'test.npz']

def has_processed_data(dataset_name, horizon):
    return all(path.exists() for path in required_data_files(dataset_name, horizon))

def result_paths(dataset_name, horizon, model_name):
    tag = f"_{RUN_CONFIG['run_tag']}" if RUN_CONFIG['run_tag'] else ''
    run_name = f'{dataset_name}_h{horizon}_{model_name}{tag}'
    out_dir = ROOT / 'results' / f'h{horizon}' / dataset_name / model_name / run_tag_dir
    return out_dir / f'{run_name}_results.npy', out_dir / f'{run_name}_summary.json'

def build_plan():
    rows = []
    for dataset_name in datasets:
        for horizon in horizons:
            data_ok = has_processed_data(dataset_name, horizon)
            missing = [str(path.relative_to(ROOT)) for path in required_data_files(dataset_name, horizon) if not path.exists()]
            for model_name in models:
                result_path, summary_path = result_paths(dataset_name, horizon, model_name)
                completed = result_path.exists() and summary_path.exists()
                status = 'completed' if completed else ('pending' if data_ok else 'missing_data')
                rows.append({
                    'dataset': dataset_name,
                    'horizon': horizon,
                    'model': model_name,
                    'status': status,
                    'result': str(result_path.relative_to(ROOT)),
                    'summary': str(summary_path.relative_to(ROOT)),
                    'missing_files': '; '.join(missing),
                })
    return rows

plan = build_plan()
counts = defaultdict(int)
for row in plan:
    counts[row['status']] += 1
print(f"实验矩阵: {len(datasets)} datasets × {len(horizons)} horizons × {len(models)} models = {len(plan)} runs")
print(dict(counts))
print('已注册验证集最优配置:')
for model_name in models:
    cfg = MODEL_CONFIGS[model_name]
    metric = cfg['selection_metric']
    print(
        f"  {model_name}: val_loss={metric['best_val_loss']:.6f}, "
        f"train={cfg['training']}, model={cfg['model']}"
    )

try:
    import pandas as pd
    plan_df = pd.DataFrame(plan)
    display(plan_df.groupby(['dataset', 'status']).size().reset_index(name='runs'))
    display(plan_df.head(20))
except ImportError:
    for row in plan[:20]:
        print(row)

## 2. 可选：补齐缺失的预处理数据

如果 `PREPARE_MISSING_DATA=True`，下面单元会调用项目已有预处理函数，仅为缺失的数据集/步长生成 `data/processed/` 文件。ECL 特征数多，完整预处理和后续训练都可能耗时、占空间。

In [ ]:
from scripts.preprocess_data import DATASET_CONFIG, preprocess_dataset

missing_by_dataset = defaultdict(set)
for dataset_name in datasets:
    for horizon in horizons:
        if not has_processed_data(dataset_name, horizon):
            missing_by_dataset[dataset_name].add(horizon)

if not missing_by_dataset:
    print('预处理数据完整。')
elif not PREPARE_MISSING_DATA:
    print('发现缺失预处理数据，但 PREPARE_MISSING_DATA=False，本次不会自动生成。')
    for dataset_name, hs in sorted(missing_by_dataset.items()):
        print(f'  {dataset_name}: {sorted(hs)}')
    if MISSING_DATA_POLICY == 'raise':
        raise FileNotFoundError('存在缺失预处理数据；请先补齐，或将 MISSING_DATA_POLICY 改为 skip。')
else:
    max_samples = PREPROCESS_MAX_SAMPLES_PER_SPLIT if PREPROCESS_MAX_SAMPLES_PER_SPLIT > 0 else None
    for dataset_name, hs in sorted(missing_by_dataset.items()):
        if dataset_name not in DATASET_CONFIG:
            raise ValueError(f'Unknown dataset for preprocessing: {dataset_name}')
        print(f'开始预处理 {dataset_name}: horizons={sorted(hs)}, max_samples_per_split={max_samples}')
        preprocess_dataset(
            dataset_name,
            sorted(hs),
            lookback=96,
            force=False,
            output_dir=data_dir,
            max_samples_per_split=max_samples,
        )
    plan = build_plan()
    print('预处理检查完成，请重新运行上一节计划单元查看最新状态。')

## 3. 运行训练

这个单元可以随时停止。已经完整保存的实验会被跳过；被中断的当前实验下次会重新跑。进度状态会写入 `results/run_state/<run_tag>_state.json`。

In [ ]:
from scripts.run_experiments import run_one

def args_for_model(model_name):
    model_args = SimpleNamespace(**RUN_CONFIG)
    training_cfg = MODEL_CONFIGS[model_name]['training']
    for key, value in training_cfg.items():
        setattr(model_args, key, value)
    model_args.skip_existing = True
    return model_args

args = SimpleNamespace(**RUN_CONFIG)
args.skip_existing = True
state_dir = ROOT / 'results' / 'run_state'
state_dir.mkdir(parents=True, exist_ok=True)
state_path = state_dir / f"{run_tag_dir}_state.json"

plan = build_plan()
pending = [row for row in plan if row['status'] == 'pending']
missing = [row for row in plan if row['status'] == 'missing_data']
completed = [row for row in plan if row['status'] == 'completed']

if missing:
    print(f'缺失数据的实验数: {len(missing)}')
    if MISSING_DATA_POLICY == 'raise':
        raise FileNotFoundError('存在 missing_data 实验，请先运行预处理单元。')
    print('MISSING_DATA_POLICY=skip，将跳过这些实验。')

if RUN_ONLY_FIRST_N is not None:
    pending = pending[:RUN_ONLY_FIRST_N]

print(f'已完成: {len(completed)}，待运行: {len(pending)}，缺失数据跳过: {len(missing)}')

def write_state(status, row=None, index=None, total=None, error=None):
    payload = {
        'status': status,
        'run_tag': RUN_CONFIG['run_tag'],
        'updated_at': time.strftime('%Y-%m-%d %H:%M:%S'),
        'completed_before_start': len(completed),
        'pending_this_run': len(pending),
        'missing_data': len(missing),
        'current_index': index,
        'current_total': total,
        'current_experiment': row,
        'current_model_config': MODEL_CONFIGS.get(row['model']) if row else None,
        'error': error,
    }
    state_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')

write_state('started')
try:
    for idx, row in enumerate(pending, start=1):
        model_args = args_for_model(row['model'])
        print('\n' + '=' * 100)
        print(f"[{idx}/{len(pending)}] {row['dataset']} h{row['horizon']} {row['model']}")
        print(f"training config: lr={model_args.lr}, weight_decay={model_args.weight_decay}, epochs={model_args.epochs}, patience={model_args.patience}, batch_size={model_args.batch_size}")
        print(f"model config: {MODEL_CONFIGS[row['model']]['model']}")
        write_state('running', row=row, index=idx, total=len(pending))
        run_one(model_args, row['dataset'], int(row['horizon']), row['model'])
        write_state('completed_one', row=row, index=idx, total=len(pending))
    write_state('finished')
    print('\n全部可运行实验已完成。')
except KeyboardInterrupt:
    write_state('interrupted', error='KeyboardInterrupt')
    print('\n训练已中断。重新运行本单元会跳过已完成实验，并从未完成项继续。')
except Exception as exc:
    write_state('failed', error=repr(exc))
    raise

## 4. 汇总已完成结果

这个单元可在训练中途或结束后随时运行。它只读取当前 `run_tag` 下已经写出的 summary。

In [ ]:
from scripts.summarize_results import load_rows, format_markdown_table

try:
    import pandas as pd
except ImportError as exc:
    raise ImportError('请先执行 pip install -r requirements.txt 安装 pandas 后再运行汇总单元。') from exc

rows = load_rows(
    ROOT / 'results',
    set(datasets),
    set(horizons),
    set(models),
    {RUN_CONFIG['run_tag']},
)

if not rows:
    print('当前 run_tag 尚无已完成 summary。')
else:
    df = pd.DataFrame(rows).sort_values(['dataset', 'horizon', 'model'])
    out_csv_dir = ROOT / 'results' / 'v2_csv' / 'full_matrix'
    out_md_dir = ROOT / 'results' / 'v2_md' / 'full_matrix'
    out_csv_dir.mkdir(parents=True, exist_ok=True)
    out_md_dir.mkdir(parents=True, exist_ok=True)
    csv_path = out_csv_dir / f"{RUN_CONFIG['run_tag']}_summary.csv"
    md_path = out_md_dir / f"{RUN_CONFIG['run_tag']}_summary.md"
    df.to_csv(csv_path, index=False)
    md_path.write_text(format_markdown_table(df), encoding='utf-8')
    print(f'已完成 summary 数: {len(df)} / {len(plan)}')
    print(f'CSV: {csv_path.relative_to(ROOT)}')
    print(f'MD:  {md_path.relative_to(ROOT)}')
    display(df[['dataset', 'horizon', 'model', 'MSE', 'MAE', 'R2', 'MSE_target', 'R2_target', 'trained_epochs', 'train_time_seconds']].head(30))

## 5. 查看剩余任务

训练中断后运行本单元，可以快速确认还剩哪些实验。

In [ ]:
plan = build_plan()
try:
    import pandas as pd
    plan_df = pd.DataFrame(plan)
    display(plan_df.groupby('status').size().reset_index(name='runs'))
    display(plan_df[plan_df['status'] != 'completed'].head(50))
except ImportError:
    print(defaultdict(int, ((status, sum(1 for row in plan if row['status'] == status)) for status in {'completed', 'pending', 'missing_data'})))
    for row in plan:
        if row['status'] != 'completed':
            print(row)